In [ ]:
from networkx.classes import non_edges
from nltk.translate.lepor import length_penalty
from pydantic.v1.utils import truncate
from sympy.physics.units import temperature
!pip install "transformers>=4.41" datasets accelerate peft evaluate rouge_score scikit-learn sentencepiece

In [ ]:
import os
from collections import Counter, OrderedDict
from dataclasses import dataclass
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
import evaluate
import numpy as np
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

In [ ]:
import pandas as pd
from pathlib import Path
from os import getcwd
current_dir = getcwd()
LOCAL_DATA_PATH= Path(current_dir).parent / ".." / "datafiles" / "yelp_reviews_clean_CA.csv"
try:
    review_dataset = pd.read_csv(filepath_or_buffer=LOCAL_DATA_PATH,
                                 on_bad_lines="skip",
                                 encoding="utf-8",
                                 )
except FileNotFoundError as e:
    print(e)

training_data,validation_data = train_test_split(review_dataset, test_size=0.2, random_state=7,stratify=review_dataset['sentiment'])
training_data = training_data.to_dict(orient="records")
validation_data = validation_data.to_dict(orient="records")
def token_building_method(tokenizer, max_source_len, max_target_len):
   def builder_method(batch):
       inputs = tokenizer(
           batch["input"],
           max_length=max_source_len,
           truncation=True,
       )
       labels = tokenizer(
           text_target = batch["target"],
           truncation=True,
           max_length=max_target_len,
       )
       inputs["labels"] = [
           [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in seq]
           for seq in labels["input_ids"]
       ]
       return inputs
   return builder_method

In [ ]:
@dataclass
class FlanT5Config:
    model: str = "google/flan-t5-base"
    training_data:str = LOCAL_DATA_PATH
    eval_data:str = LOCAL_DATA_PATH
    output_dir:str = "./flant5-reviews"
    max_source_len: int = 512
    max_target_len: int = 256
    max_reviews_per_business: int = 30
    epochs: int = 2
    learning_rate: float = 5e-3
    batch_size:int = 16
    gradient_accumulation_steps: int = 8
    fine_tune_model: bool = False
    seed: int = 7
MODEL_ARGS = FlanT5Config()
MODEL_ARGS

### Instruction Definition Block


Since FLAN-T5 is an instruction based model, this block serves to define the instruction templates that will be used during training and infrencing. This section will also provide template response, since FLAN-T5 is an decoder-encoder style model there is no traditional classification, rather it is generating the sentiment which we will evaluate during training to see if they match

- `SENTIMENT_PROMPT` will be used to prompt the model to generate a "classification" that matches the provided reviews
- `REVIEW_SUMMARY_PROMPT` will be used to generate a short summary based on the provided reviews that will help inform users of our site with the general overview of other customer experiences. Highlighting any issues or concerns.

In [ ]:
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
SENTIMENT_PROMPT = (
    "classify the sentiment of this review, using negative, neutral, or positive. review: {input_review} \n sentiment class:"
)
REVIEW_SUMMARY_PROMPT = (
    "summarize the following reviews: {reviews}.\n"
    "business: {business}\n"
    "include the overall sentiment distribution provided by: {sentiment_distro}.\n"
    "Summary:"
)

In [ ]:
def label_dist(labels):
    labels = [tag for tag in labels if tag in SENTIMENT_LABELS]
    num_labels = len(labels)
    count = Counter(labels)
    result_percentage = {k: (round(100*count.get(k, 0)/num_labels) if num_labels else 0) for k in SENTIMENT_LABELS}
    return num_labels, result_percentage

def print_results(num_labels: int, result_percentage: dict) ->str:
    if num_labels == 0:
        return "No reviews with valid labels were detected"
    return(f"{num_labels}: reviews_detected, result_percentage_positive:{result_percentage['positive']} \n result_percentage_negative:{result_percentage['negative']} \n \
     results_precentage_neutral:{result_percentage['neutral']}  labels were detected")

def filter_reviews(reviews, tokenizer, review_limit, token_limit=256, review_length_limit=(MODEL_ARGS.max_source_len-len(REVIEW_SUMMARY_PROMPT))):
    filtered_output = []
    num_selected = 0
    for review in reviews[:review_limit]:
        review = (review or "").strip()
        if not review:
            continue
        ids = tokenizer(review, truncation=True, max_length=review_length_limit)['input_ids']
        if num_selected + len(ids) > token_limit and filtered_output:
            break
        filtered_output.append(tokenizer.decode(ids, skip_special_tokens=True).strip())
        num_selected += len(ids)
    return filtered_output

In [ ]:
def t5_multitask(rows, tokenizer=None, review_limit=30, review_length_limit=150):
    inputs = []
    targets = []
    tasks = []
    groups = OrderedDict()

    for row in rows:
        input = (row.get("text") or "").strip()
        if not input:
            continue
        label = (row.get("sentiment") or "").strip()
        if label in SENTIMENT_LABELS:
            inputs.append(SENTIMENT_PROMPT.format(input_review=input))
            targets.append(label)
            tasks.append("sentiment")
        business = (row.get("business_name") or "").strip()
        if business:
            groups.setdefault(business, []).append({
                "text": input,
                "label": label,
                "buisness_review_summary": "",
            })

    for business, reviews in groups.items():
        sum = next((entry['buisness_review_summary'] for entry in reviews if entry['buisness_review_summary']), "")
        if not sum:
            continue
        filtered_reviews = filter_reviews([entry['text'] for entry in reviews], tokenizer, review_limit, review_length_limit)
        num_reviews, percentage = label_dist([entry['sentiment'] for entry in reviews])
        review_blob = ".".join(f"{text}" for text in filtered_reviews)
        inputs.append(REVIEW_SUMMARY_PROMPT.format(
            buisness=business,
            reviews=review_blob,
            sentiment_distro=print_results(num_reviews, percentage),
        ))
        targets.append(label)
        tasks.append("summary")
    return Dataset.from_dict({"input": inputs, "target": targets, "task": tasks})


In [ ]:
@torch.inference_mode()
def sentiment_classifier(review, tokenizer, model, device, max_length= MODEL_ARGS.max_target_len):
    encoded = tokenizer(SENTIMENT_PROMPT.format(input_review=review), return_tensors="pt", truncate=True, max_lenth=max_length).to(device)
    output = model.generate(**encoded, max_new_tokens=5, num_beams=4)
    sentiment = tokenizer.decode(output[0], skip_special_tokens=True).strip().lower()
    print(sentiment)
    return sentiment if sentiment in SENTIMENT_LABELS else "neutral"

@torch.inference_mode()
def business_review_summarizer(business_name, reviews, model, tokenizer, device, max_reviews=30, max_source_len=MODEL_ARGS.max_source_len | 512, review_length_limit=MODEL_ARGS.max_target_len):
    labels = [sentiment_classifier(review, tokenizer, model, device) for review in reviews]
    num_reviews, percentage = label_dist(labels)
    filtered_reviews = filter_reviews(reviews, tokenizer, review_limit=max_reviews, review_length_limit=review_length_limit)
    review_blob = "\n".join(f"-{text}" for text in filtered_reviews)
    summary_prompt = REVIEW_SUMMARY_PROMPT.format(
        business=business_name,
        reviews=review_blob,
        sentiment_distro=print_results(num_reviews, percentage),
    )
    print(repr(summary_prompt))
    encoded = tokenizer(summary_prompt, return_tensors="pt", truncation=True, max_length=max_source_len).to(device)
    output = model.generate(**encoded,max_new_tokens=review_length_limit,num_beams=4,no_repeat_ngram_size=3,length_penalty=2.5, repetition_penalty=2.0, do_sample=True, temperature=0.8)
    review_summary = tokenizer.decode(output[0], skip_special_tokens=True).strip()
    return {"business": business_name, "number_of_reviews":num_reviews, "sentiment_composition": percentage, "summary": review_summary}

In [ ]:
set_seed(MODEL_ARGS.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device: ", device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ARGS.model)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ARGS.model)
model.to(device)
print(f"Loaded: {MODEL_ARGS.model}")

In [ ]:
# Zero-shot summarization on the freshly loaded base model (NO training yet).
from pprint import pprint

model.eval()
zeroshot_business = "Sample Cafe"
zeroshot_reviews = [
    "Great coffee and cozy seating, but service was slow at lunch.",
    "Overpriced pastries and the barista got my order wrong twice.",
    "Nice atmosphere but the tables were dirty and the music was too loud.",
    "Great food, and the ice coffee was tasty!",
    "Parking was a challenge but overall the experience was a nice one",
    "Lines were long, and it took forever to get my order"
]

print("ZERO-SHOT (base model, no fine-tuning):\n")
pprint(business_review_summarizer(
    zeroshot_business, zeroshot_reviews, model, tokenizer, device,
    max_reviews=MODEL_ARGS.max_reviews_per_business, max_source_len=MODEL_ARGS.max_source_len,
))
print(
    "\nRead the summary above. If it's good enough, you can stop here and skip "
    "summary fine-tuning\n(leave business_summary blank -> build_multitask trains "
    "sentiment only). If not, that's\nyour reason to fine-tune, and this output is a "
    "starting point for bootstrapped targets."
)
print(model.name_or_path)
print(repr(business_review_summarizer))

In [ ]:
assert os.path.exists(MODEL_ARGS.training_data),(
    f"training data not found"
)

if MODEL_ARGS.fine_tune_model:
    from peft import LoraConfig, TaskType, get_peft_model
    model = get_peft_model(
        model,
        LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            r=16, lora_alpha=32,lora_dropout=0.05,
            target_modules= ["q","v"],
        )
    )
    model.print_trainable_parameters()
    if MODEL_ARGS.learning_rate < 1e-3:
        print("learning rate too low for fine tuning of the model")
tokenizer_fn = token_building_method(tokenizer,MODEL_ARGS.max_source_len,MODEL_ARGS.max_target_len)
training_data = t5_multitask(training_data, tokenizer, MODEL_ARGS.max_reviews_per_business, MODEL_ARGS.max_source_len)
training_data = training_data.shuffle(seed=MODEL_ARGS.seed).map(tokenizer_fn, batched=True, remove_columns=training_data.column_names)

if MODEL_ARGS.eval_data and os.path.exists(MODEL_ARGS.eval_data):
    validation_data = t5_multitask(validation_data, tokenizer, MODEL_ARGS.max_reviews_per_business,MODEL_ARGS.max_source_len)
    validation_data = validation_data.map(tokenizer_fn, batched=True, remove_columns=validation_data.column_names)

sample_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100,pad_to_multiple_of=8)

bf16_check = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_parameters = Seq2SeqTrainingArguments(
    output_dir=MODEL_ARGS.output_dir,
    per_device_train_batch_size=MODEL_ARGS.batch_size,
    per_device_eval_batch_size=MODEL_ARGS.batch_size,
    gradient_accumulation_steps=MODEL_ARGS.gradient_accumulation_steps,
    learning_rate=MODEL_ARGS.learning_rate,
    num_train_epochs=MODEL_ARGS.epochs,
    bf16=bf16_check,
    fp16=False,
    gradient_checkpointing=True,
    predict_with_generate=True,
    generation_max_length=MODEL_ARGS.max_target_len,
    generation_num_beams=4,
    logging_steps=5
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_parameters,
    train_dataset=training_data,
    eval_dataset=validation_data,
    data_collator=sample_collator,
)

trainer.train()
trainer.save_model(MODEL_ARGS.output_dir)